# Vaccination Data Analysis and Visualization

**Domain:** Public Health and Epidemiology  
**Tools:** Python, Pandas, SQLite/SQL, Plotly, Streamlit, Power BI-ready CSV exports

This notebook implements the supplied project brief: data cleaning, EDA, SQL database creation, analytical questions, exports for Power BI, and files for a Streamlit dashboard. The project brief defines five source tables: coverage, incidence rate, reported cases, vaccine introduction, and vaccine schedule. fileciteturn0file0L125-L190

**Important:** The notebook first looks for the supplied dataset in Google Drive or `/content/data`. If the source files are not available in the runtime, it creates a clearly-labelled demo dataset so every cell can still be executed. Replace the demo files with the real project files before final submission.

In [ ]:
!pip -q install pandas numpy openpyxl plotly seaborn sqlalchemy gdown
import os, re, glob, sqlite3, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display
warnings.filterwarnings('ignore')

BASE = Path('/content/vaccination_project')
DATA_DIR = BASE / 'data'
EXPORT_DIR = BASE / 'exports'
MODEL_DIR = BASE / 'models'
for p in [DATA_DIR, EXPORT_DIR, MODEL_DIR]: p.mkdir(parents=True, exist_ok=True)
print('Project directory:', BASE)


## 1. Get the supplied dataset

The project brief gives a Google Drive folder as the dataset source. fileciteturn0file0L123-L126

Run the next cell. If the shared folder is publicly downloadable, `gdown` will copy its contents. Otherwise, manually upload the CSV/XLSX files into `/content/vaccination_project/data`.

In [ ]:
DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1YQ6mNrZCrlEeBP4GH3VnLBNXb7OBD4tf?usp=sharing'
!gdown --folder "$DRIVE_FOLDER_URL" -O /content/vaccination_project/data --quiet || true

files = [str(p) for p in DATA_DIR.rglob('*') if p.is_file()]
print('Files found:', len(files))
for f in files[:30]: print(f)


In [ ]:
def read_any(path):
    path = Path(path)
    if path.suffix.lower() == '.csv': return pd.read_csv(path, low_memory=False)
    if path.suffix.lower() in ['.xlsx','.xls']:
        return pd.read_excel(path)
    return None

raw_files = [p for p in DATA_DIR.rglob('*') if p.suffix.lower() in ['.csv','.xlsx','.xls']]
raw = {}
for p in raw_files:
    try:
        raw[p.stem.lower()] = read_any(p)
    except Exception as e:
        print('Could not read', p, e)
print({k:v.shape for k,v in raw.items()})


## 2. Flexible table detection and schema standardization

The supplied specification names the five tables and their important fields. Coverage includes target population, doses and coverage; incidence contains incidence rate; reported cases contains cases; vaccine introduction contains WHO region/year/introduction; and schedule contains rounds and target population. fileciteturn0file0L128-L190

In [ ]:
ALIASES = {
 'coverage': ['coverage','coverage_data','vaccination_coverage','table1'],
 'incidence': ['incidence','incidence_rate','incidence_rate_data','table2'],
 'cases': ['cases','reported_cases','reported_case','table3'],
 'introduction': ['introduction','vaccine_introduction','vaccine introduction','table4'],
 'schedule': ['schedule','vaccine_schedule','vaccine schedule','table5']
}

def norm_name(s): return re.sub(r'[^a-z0-9]+','_',str(s).strip().lower()).strip('_')
def standardize_columns(df):
    df = df.copy()
    df.columns = [norm_name(c) for c in df.columns]
    ren = {}
    for c in df.columns:
        if c in ['country','country_name','name']: ren[c] = 'name'
        elif c in ['iso_3_code','iso3','iso_alpha_3','code']: ren[c] = 'code'
        elif c in ['year','yr']: ren[c] = 'year'
        elif c in ['target_number','target_pop','target_population']: ren[c] = 'target_population'
        elif c in ['dodge','doses','dose','doses_administered']: ren[c] = 'doses'
        elif c in ['coverage_pct','coverage_percent','vaccination_coverage']: ren[c] = 'coverage'
        elif c in ['incidence_rate','incidence']: ren[c] = 'incidence_rate'
        elif c in ['reported_cases','case','cases']: ren[c] = 'cases'
        elif c in ['who_region','region']: ren[c] = 'who_region'
        elif c in ['intro','introduced']: ren[c] = 'intro'
        elif c in ['antigen_description','vaccine_description']: ren[c] = 'description'
    return df.rename(columns=ren)

tables = {k: None for k in ALIASES}
for key, aliases in ALIASES.items():
    for name, df in raw.items():
        n = norm_name(name)
        if any(a.replace(' ','_') in n for a in aliases):
            tables[key] = standardize_columns(df)
            break

for k,v in tables.items():
    if v is not None: print(k, v.shape, list(v.columns))


## 3. Demo fallback (only when source files are unavailable)

This section is intentionally synthetic. It is not a replacement for the supplied source data; it only makes the notebook runnable end-to-end when the Drive folder is inaccessible.

In [ ]:
def make_demo_tables(seed=42):
    rng = np.random.default_rng(seed)
    countries = [('IND','India','SEAR'),('USA','United States','AMR'),('BRA','Brazil','AMR'),('NGA','Nigeria','AFR'),('DEU','Germany','EUR'),('JPN','Japan','WPR'),('GBR','United Kingdom','EUR'),('KEN','Kenya','AFR'),('AUS','Australia','WPR'),('MEX','Mexico','AMR')]
    years = np.arange(2015,2025)
    antigens = [('BCG','BCG'),('HEPB','Hepatitis B'),('MCV1','Measles-containing vaccine'),('POL3','Polio')]
    cov=[]; inc=[]; cases=[]; intro=[]; sched=[]
    for code,name,region in countries:
      for y in years:
        base = 55 + (hash(code)%20) + (y-2015)*1.8 + rng.normal(0,4)
        for ag,desc in antigens:
          coverage=float(np.clip(base+rng.normal(0,5),20,99))
          target=int(rng.integers(50000,500000))
          cov.append([region,code,name,y,ag,desc,'official','Official estimate',target,int(target*coverage/100),coverage])
        measles=max(0, 1800-(base*18)+rng.normal(0,120))
        inc.append([region,code,name,y,'MEAS','Measles','100000 population',measles/10])
        cases.append([region,code,name,y,'MEAS','Measles',int(measles)])
        intro.append([code,name,region,y,'Measles vaccine',int(y>=2017)])
        sched.append([code,name,region,y,'MCV1','Measles-containing vaccine',1,'Children', 'National', '9 months','Demo'])
    return (pd.DataFrame(cov,columns=['group','code','name','year','antigen','description','coverage_category','coverage_category_description','target_population','doses','coverage']),
            pd.DataFrame(inc,columns=['group','code','name','year','disease','disease_description','denominator','incidence_rate']),
            pd.DataFrame(cases,columns=['group','code','name','year','disease','disease_description','cases']),
            pd.DataFrame(intro,columns=['iso_3_code','country_name','who_region','year','description','intro']),
            pd.DataFrame(sched,columns=['iso_3_code','country_name','who_region','year','vaccine_code','vaccine_description','schedule_rounds','target_pop_description','geoarea','age_administered','source_comment']))

if any(v is None for v in tables.values()):
    print('One or more source tables were not detected. Creating DEMO data.')
    demo = make_demo_tables()
    tables = dict(zip(tables.keys(), demo))
    USING_DEMO = True
else:
    USING_DEMO = False
print('USING_DEMO =', USING_DEMO)

coverage, incidence, cases, introduction, schedule = tables.values()


## 4. Data cleaning and quality checks

The project brief specifically requires missing-value handling, consistent units, and uniform date/year formatting. fileciteturn0file0L23-L29

In [ ]:
def clean_table(df):
    df=df.copy()
    df.columns=[norm_name(c) for c in df.columns]
    for c in df.columns:
        if c in ['year','target_population','doses','coverage','incidence_rate','cases','schedule_rounds','intro']:
            df[c]=pd.to_numeric(df[c], errors='coerce')
    if 'year' in df: df['year']=df['year'].round().astype('Int64')
    if 'coverage' in df: df['coverage']=df['coverage'].clip(lower=0,upper=100)
    return df.drop_duplicates().reset_index(drop=True)

coverage, incidence, cases, introduction, schedule = [clean_table(x) for x in [coverage,incidence,cases,introduction,schedule]]

quality = []
for name,df in [('coverage',coverage),('incidence',incidence),('cases',cases),('introduction',introduction),('schedule',schedule)]:
    quality.append({'table':name,'rows':len(df),'columns':len(df.columns),'duplicate_rows':int(df.duplicated().sum()),'missing_cells':int(df.isna().sum().sum())})
quality_df=pd.DataFrame(quality)
display(quality_df)


## 5. Exploratory Data Analysis

The required analysis includes vaccination trends, disease incidence, regional disparities and correlation analysis. fileciteturn0file0L54-L56

In [ ]:
year_cov=coverage.groupby('year',dropna=True)['coverage'].mean().reset_index()
fig=px.line(year_cov,x='year',y='coverage',markers=True,title='Average Vaccination Coverage Over Time',labels={'coverage':'Average coverage (%)'})
fig.show()

if {'who_region','year','coverage'}.issubset(coverage.columns):
    region_cov=coverage.groupby(['who_region','year'])['coverage'].mean().reset_index()
    px.line(region_cov,x='year',y='coverage',color='who_region',markers=True,title='Vaccination Coverage by WHO Region').show()

if {'year','cases'}.issubset(cases.columns):
    cases_year=cases.groupby('year')['cases'].sum().reset_index()
    px.line(cases_year,x='year',y='cases',markers=True,title='Reported Disease Cases Over Time').show()


## 6. Answer the required analytical questions

The following cells produce reusable tables for the questions listed in the project brief, including dose drop-off, gender/education/urban-rural comparisons where those dimensions exist, booster trends, population density, high-incidence/high-coverage areas, vaccine introduction effects, and coverage gaps. fileciteturn0file0L58-L100

**Note:** Gender, education, urban/rural, population density, season/month and vaccination strategy are not listed as columns in the supplied five-table schema. They cannot be answered from those tables unless the actual source files contain additional fields. The notebook therefore checks for them rather than inventing results.

In [ ]:
# Q1/Q9: vaccination coverage vs disease incidence
if {'code','year','coverage'}.issubset(coverage.columns) and {'code','year','incidence_rate'}.issubset(incidence.columns):
    c=coverage.groupby(['code','year'])['coverage'].mean().reset_index()
    i=incidence.groupby(['code','year'])['incidence_rate'].mean().reset_index()
    corr_df=c.merge(i,on=['code','year']).dropna()
    print('Correlation (coverage vs incidence):', corr_df['coverage'].corr(corr_df['incidence_rate']))
else: print('Q1/Q9: required fields unavailable.')

# Q2: first dose vs subsequent dose drop-off (using schedule rounds if available)
if 'schedule_rounds' in schedule.columns:
    rounds=schedule.groupby('schedule_rounds').size().reset_index(name='records').sort_values('schedule_rounds')
    print(rounds)
else: print('Q2: schedule rounds unavailable.')

# Q6: booster trend (search description/code fields for booster)
booster_cols=[c for c in coverage.columns if c in ['antigen','description']]
if booster_cols:
    mask=False
    for c in booster_cols: mask = mask | coverage[c].astype(str).str.contains('booster',case=False,na=False)
    print(coverage.loc[mask].groupby('year')['coverage'].mean().reset_index())
else: print('Q6: booster identifier not available.')

# Q10: high incidence despite high coverage
if {'code','year','coverage'}.issubset(coverage.columns) and {'code','year','incidence_rate'}.issubset(incidence.columns):
    m=coverage.groupby(['code','year'])['coverage'].mean().reset_index().merge(incidence.groupby(['code','year'])['incidence_rate'].mean().reset_index(),on=['code','year'])
    high=m[(m.coverage>=80) & (m.incidence_rate>=m.incidence_rate.quantile(.75))].sort_values('incidence_rate',ascending=False)
    display(high.head(20))

# Q3/Q4/Q5/Q7/Q8: only if actual source contains these fields
for optional in ['gender','education_level','urban_rural','population_density','month','date','season']:
    available=[t for t,df in [('coverage',coverage),('incidence',incidence),('cases',cases),('introduction',introduction),('schedule',schedule)] if optional in df.columns]
    print(optional, '->', available if available else 'not present in supplied schema')


In [ ]:
# Medium-level: vaccine introduction vs disease cases
if {'code','year','intro'}.issubset(introduction.columns) and {'code','year','cases'}.issubset(cases.columns):
    intro_year=introduction[introduction['intro'].fillna(0)>0].groupby('code')['year'].min().rename('intro_year').reset_index()
    c=cases.merge(intro_year,on='code',how='inner')
    c['period']=np.where(c['year']<c['intro_year'],'Before','After')
    print(c.groupby('period')['cases'].mean())
    px.box(c,x='period',y='cases',title='Reported Cases Before vs After Vaccine Introduction').show()

# Coverage target percentage by antigen
if {'antigen','target_population','doses'}.issubset(coverage.columns):
    target=coverage.groupby('antigen')[['target_population','doses']].sum().reset_index()
    target['target_coverage_pct']=100*target['doses']/target['target_population'].replace(0,np.nan)
    display(target.sort_values('target_coverage_pct',ascending=False))


## 7. SQL database creation

The brief requires relational tables, normalization, and primary/foreign-key integrity. fileciteturn0file0L31-L35

SQLite is used in Colab because it needs no server. The resulting `.db` file can be inspected with SQL and the cleaned CSV exports can be loaded into a production MySQL/PostgreSQL database later.

In [ ]:
DB_PATH=BASE/'vaccination.db'
conn=sqlite3.connect(DB_PATH)
for name,df in [('coverage',coverage),('incidence_rate',incidence),('reported_cases',cases),('vaccine_introduction',introduction),('vaccine_schedule',schedule)]:
    df.to_sql(name,conn,if_exists='replace',index=False)
queries={
 'coverage_by_year': "SELECT year, ROUND(AVG(coverage),2) AS avg_coverage FROM coverage GROUP BY year ORDER BY year",
 'regional_coverage': "SELECT year, group, ROUND(AVG(coverage),2) AS avg_coverage FROM coverage GROUP BY year, group ORDER BY year",
 'disease_cases': "SELECT year, disease, SUM(cases) AS total_cases FROM reported_cases GROUP BY year,disease ORDER BY year",
}
for q,sql in queries.items():
    print('\n',q); display(pd.read_sql_query(sql,conn).head(20))
conn.close()
print('SQLite database:',DB_PATH)


## 8. Export cleaned data for Power BI

The project brief calls for Power BI to connect to the cleaned SQL data and visualize coverage, incidence and antigen trends. fileciteturn0file0L37-L51

In [ ]:
exports={'coverage_clean':coverage,'incidence_rate_clean':incidence,'reported_cases_clean':cases,'vaccine_introduction_clean':introduction,'vaccine_schedule_clean':schedule,'data_quality_report':quality_df}
for name,df in exports.items(): df.to_csv(EXPORT_DIR/f'{name}.csv',index=False)
print('Exported files:')
for p in sorted(EXPORT_DIR.glob('*.csv')): print(p.name, p.stat().st_size, 'bytes')


## 9. Executive insights table

Use this output as the starting point for the Power BI dashboard and final documentation. Do not describe correlation as causation without a suitable causal study design.

In [ ]:
insights=[]
insights.append(['Average vaccination coverage', round(float(coverage['coverage'].mean()),2) if 'coverage' in coverage else np.nan])
if 'coverage' in coverage:
    by_country=coverage.groupby('name')['coverage'].mean().sort_values()
    insights.append(['Lowest average coverage country', by_country.index[0] if len(by_country) else 'N/A'])
    insights.append(['Highest average coverage country', by_country.index[-1] if len(by_country) else 'N/A'])
if 'cases' in cases: insights.append(['Total reported cases', int(cases['cases'].sum())])
if {'coverage','incidence_rate'}.issubset(set(corr_df.columns) if 'corr_df' in globals() else set()): insights.append(['Coverage-incidence correlation', round(float(corr_df['coverage'].corr(corr_df['incidence_rate'])),4)])
insight_df=pd.DataFrame(insights,columns=['Metric','Value'])
display(insight_df)
insight_df.to_csv(EXPORT_DIR/'executive_insights.csv',index=False)


## 10. Download project outputs

The notebook creates a SQLite database and Power BI-ready CSVs under `/content/vaccination_project/exports`. The GitHub package in this project contains the Streamlit application and supporting files.

In [ ]:
import shutil
shutil.make_archive('/content/vaccination_project_outputs','zip','/content/vaccination_project')
print('/content/vaccination_project_outputs.zip')
